# research-ir — демо

**Перевод функциональных языков через общий IR + NLP, без LLVM.**

Идея из *Code Translation with Compiler Representations* (ICLR 2023), где C++/Java/Rust/Go
переводят через LLVM IR. Здесь — Haskell и Lean, но вместо LLVM используем **нативные IR
компиляторов** (GHC Core, Lean `Expr`), нормализованные в один общий IR.

```
исходник --frontend--> нативный IR --> общий IR --[seq2seq]--> код на другом языке
```

Запусти **Run All**. Ниже два результата: (1) один IR для двух компиляторов, (2) обучение перевода.

In [1]:
import os, sys, json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'ml'))
print('ROOT =', ROOT)

ROOT = /Users/aizat/projects/hackathons/research-ir


## Результат 1 — один IR для двух разных компиляторов

Функция `inc n = Succ n` независимо прошла через GHC Core (Haskell) и Lean `Expr`.
Имена различаются, но **структурный скелет обязан совпасть** — это и есть «общий IR-пивот».

In [2]:
from research_ir.ir import loads, shape, dumps, Module

def get_def(fixture, name):
    mod = loads(Path(fixture).read_text())
    d = next(d for d in mod.defs if d.name == name)
    return d

hs = get_def('fixtures/add.cir', 'inc')        # из GHC Core
ln = get_def('fixtures/lean_inc.cir', 'inc')   # из Lean Expr

print('Haskell (GHC Core):', dumps(Module((hs,))))
print('Lean    (Expr)    :', dumps(Module((ln,))))
print()
print('Структурный скелет совпадает:', shape(hs) == shape(ln))

Haskell (GHC Core): (module (def inc (const Nat->Nat) (lam (n (const Nat)) (app (const Succ) (var n)))))
Lean    (Expr)    : (module (def inc (const _ty) (lam (n (const MyNat)) (app (const succ) (var _bv0)))))

Структурный скелет совпадает: True


## Результат 2 — обучение перевода IR → код (главное)

### 2a. Собираем датасет

Корпус из ~46 функций (Nat-арифметика, Bool-логика, Nat→Bool) гоним через **оба**
frontend'а и получаем пары `(Haskell-IR, Lean-код)`. (Требуется GHC и Lean в системе.)

In [3]:
# Свежий процесс через just -> всегда берёт актуальный ml/corpus.py
# (импорт build_dataset в долгоживущем ядре кэширует старый corpus -> датасет «застревает» на 17).
!just dataset

PYTHONPATH=. poetry run python ml/build_dataset.py
wrote /Users/aizat/projects/hackathons/research-ir/ml/dataset.json with 46 pairs
  idB        ir[73] -> lean[30]
  notB       ir[146] -> lean[74]
  constTrue  ir[87] -> lean[39]
  constFalse ir[89] -> lean[41]
  andB       ir[170] -> lean[73]
  orB        ir[168] -> lean[71]
  nandB      ir[147] -> lean[48]
  norB       ir[145] -> lean[46]
  xorB       ir[181] -> lean[74]
  implyB     ir[171] -> lean[74]
  nimplyB    ir[149] -> lean[50]
  eqB        ir[180] -> lean[73]
  and3       ir[189] -> lean[51]
  or3        ir[186] -> lean[48]
  xor3       ir[189] -> lean[51]
  majority3  ir[285] -> lean[81]
  inc        ir[84] -> lean[43]
  plus2      ir[105] -> lean[58]
  plus3      ir[124] -> lean[71]
  plus4      ir[143] -> lean[84]
  plus5      ir[162] -> lean[97]
  plus6      ir[181] -> lean[110]
  plus7      ir[200] -> lean[123]
  plus8      ir[219] -> lean[136]
  idn        ir[73] -> lean[32]
  czero      ir[82] -> lean[43]
  predn      

### 2b. Один пример пары: что подаём модели и что она должна выдать

In [4]:
rows = json.loads(Path('ml/dataset.json').read_text())
ex = next(r for r in rows if r['name'] == 'add')
print('ВХОД модели — Haskell-IR:')
print(' ', ex['haskell_ir'])
print()
print('ЦЕЛЬ — код на Lean:')
print(' ', ex['lean_src'])

ВХОД модели — Haskell-IR:
  (module (def add (const Nat->Nat->Nat) (let (add (const Nat->Nat->Nat) (lam (a (const Nat)) (lam (b (const Nat)) (case (var a) (alt (pcon Zero ()) (var b)) (alt (pcon Succ (m)) (app (const Succ) (app (app (var add) (var m)) (var b)))))))) (var add))))

ЦЕЛЬ — код на Lean:
  def add (a b : MyNat) : MyNat := match a with | MyNat.zero => b | MyNat.succ m => MyNat.succ (add m b)


### 2c. Дообучаем предобученный seq2seq (t5-small)

Это **сама нейросеть**: вход — наш общий IR, выход — код Lean. Берём предобученную модель
(трансфер с уже видевшей код) и дообучаем на наших парах. Метрика — `exact match` плюс мягкая
`близость`; **held-out** split показывает генерализацию (модель эти функции при обучении не видела).

In [5]:
# Тоже свежим процессом — обучает t5 на полном датасете (46), читает ml/dataset.json.
!just train-t5

PYTHONPATH=. poetry run python ml/train_t5.py
модель: t5-small   train=37  val=9  device=cpu

дообучаем ...
  epoch   1   loss 4.1514
  epoch  10   loss 0.8045
  epoch  20   loss 0.2176
  epoch  30   loss 0.0942
  epoch  40   loss 0.0605
  epoch  50   loss 0.0465
  epoch  60   loss 0.0206
  epoch  70   loss 0.0292
  epoch  80   loss 0.0255
  epoch  90   loss 0.0160
  epoch 100   loss 0.0145
  epoch 110   loss 0.0146
  epoch 120   loss 0.0122

=== Held-out (модель эти функции НЕ видела при обучении) ===

[constFalse]  ✓ exact
  gold: def constFalse (b : Bool) : Bool := false
  pred: def constFalse (b : Bool) : Bool := false

[orB]  ~90% близость
  gold: def orB (a b : Bool) : Bool := match a with | true => true | false => b
  pred: def orB (a b : Bool) : Bool := match a with | true => true

[nimplyB]  ✓ exact
  gold: def nimplyB (a b : Bool) : Bool := andB a (notB b)
  pred: def nimplyB (a b : Bool) : Bool := andB a (notB b)

[plus2]  ✓ exact
  gold: def plus2 (n : MyNat) : MyNat := MyN

## Вывод

- **Результат 1**: нативные IR двух разных компиляторов ложатся в один общий IR (скелеты совпали).
- **Результат 2**: дообученный seq2seq переводит `Haskell-IR → Lean`. На ~46 функциях с held-out
  split модель точно переводит функции, которых **не видела** при обучении (см. вывод выше) —
  это генерализация, а не запоминание.

Главная гипотеза подтверждена: функциональные языки переносятся друг в друга через общий IR + NLP, без LLVM.